# MovieLens Data Overview

MovieLens 1M の parquet ファイルを読み込み、内容を確認する。

In [28]:
from pathlib import Path

import pandas as pd

In [29]:
DATA_DIR = Path("./data")

movies_df = pd.read_parquet(DATA_DIR / "movies.parquet")
ratings_df = pd.read_parquet(DATA_DIR / "ratings.parquet")
users_df = pd.read_parquet(DATA_DIR / "users.parquet")

In [30]:
movies_df

,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
3878,3948,Meet the Parents (2000),Comedy
3879,3949,Requiem for a Dream (2000),Drama
3880,3950,Tigerland (2000),Drama
3881,3951,Two Family House (2000),Drama


In [31]:
ratings_df

,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291
...,...,...,...,...
1000204,6040,1091,1,956716541
1000205,6040,1094,5,956704887
1000206,6040,562,5,956704746
1000207,6040,1096,4,956715648


In [32]:
users_df

,user_id,gender,age,occupation,zip_code
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455
...,...,...,...,...,...
6035,6036,F,25,15,32603
6036,6037,F,45,1,76006
6037,6038,F,56,1,14706
6038,6039,F,45,0,01060


# data preprocess

In [33]:
def missing_rate(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "missing_count": df.isna().sum(),
            "missing_rate": df.isna().mean().round(4),
        }
    ).sort_values("missing_rate", ascending=False)

missing_rate(movies_df)
missing_rate(ratings_df)
missing_rate(users_df)

,missing_count,missing_rate
user_id,0,0.0
gender,0,0.0
age,0,0.0
occupation,0,0.0
zip_code,0,0.0


In [34]:
# movies_df preprocess
movies_df_transformed = movies_df.copy()

movies_df_transformed["release_year"] = movies_df_transformed["title"].str.extract(r"\((\d{4})\)")
movies_df_transformed["release_year"] = pd.to_numeric(movies_df_transformed["release_year"], errors="coerce")

movies_df_transformed["release_decade"] = (
    (movies_df_transformed["release_year"] // 10 * 10)
    .astype("Int64")
    .astype("string")
    .str.cat(pd.Series(["s"] * len(movies_df_transformed), index=movies_df_transformed.index), na_rep=None)
)

unique_genres = sorted(
    {
        genre
        for genres in movies_df_transformed["genres"].dropna()
        for genre in genres.split("|")
    }
)

for genre in unique_genres:
    movies_df_transformed[f"genre_{genre}"] = movies_df_transformed["genres"].str.contains(
        rf"(?:^|\|){genre}(?:\||$)",
        regex=True,
    ).astype(int)

release_decade_dummies = pd.get_dummies(
    movies_df_transformed["release_decade"],
    prefix="release_decade",
).astype(int)

movies_df_transformed = pd.concat(
    [movies_df_transformed, release_decade_dummies],
    axis=1,
)

movies_df_transformed.head()

,movie_id,title,genres,release_year,release_decade,genre_Action,genre_Adventure,genre_Animation,genre_Children's,genre_Comedy,...,release_decade_1910s,release_decade_1920s,release_decade_1930s,release_decade_1940s,release_decade_1950s,release_decade_1960s,release_decade_1970s,release_decade_1980s,release_decade_1990s,release_decade_2000s
0,1,Toy Story (1995),Animation|Children's|Comedy,1995,1990s,0,0,1,1,1,...,0,0,0,0,0,0,0,0,1,0
1,2,Jumanji (1995),Adventure|Children's|Fantasy,1995,1990s,0,1,0,1,0,...,0,0,0,0,0,0,0,0,1,0
2,3,Grumpier Old Men (1995),Comedy|Romance,1995,1990s,0,0,0,0,1,...,0,0,0,0,0,0,0,0,1,0
3,4,Waiting to Exhale (1995),Comedy|Drama,1995,1990s,0,0,0,0,1,...,0,0,0,0,0,0,0,0,1,0
4,5,Father of the Bride Part II (1995),Comedy,1995,1990s,0,0,0,0,1,...,0,0,0,0,0,0,0,0,1,0


In [35]:
# ratings_df preprocess
ratings_df_transformed = ratings_df.copy()

ratings_df_transformed["rating_ge_3"] = (
    ratings_df_transformed["rating"] >= 3
).astype(int)
ratings_df_transformed["rating_ge_4"] = (
    ratings_df_transformed["rating"] >= 4
).astype(int)

ratings_df_transformed["rating_datetime"] = pd.to_datetime(
    ratings_df_transformed["timestamp"],
    unit="s",
)

ratings_df_transformed.head()

,user_id,movie_id,rating,timestamp,rating_ge_3,rating_ge_4,rating_datetime
0,1,1193,5,978300760,1,1,2000-12-31 22:12:40
1,1,661,3,978302109,1,0,2000-12-31 22:35:09
2,1,914,3,978301968,1,0,2000-12-31 22:32:48
3,1,3408,4,978300275,1,1,2000-12-31 22:04:35
4,1,2355,5,978824291,1,1,2001-01-06 23:38:11


In [36]:
pd.DataFrame(
    {
        "rating_ge_3_count": ratings_df_transformed["rating_ge_3"].value_counts().sort_index(),
        "rating_ge_3_rate": ratings_df_transformed["rating_ge_3"].value_counts(normalize=True).sort_index().round(4),
        "rating_ge_4_count": ratings_df_transformed["rating_ge_4"].value_counts().sort_index(),
        "rating_ge_4_rate": ratings_df_transformed["rating_ge_4"].value_counts(normalize=True).sort_index().round(4),
    }
)

,rating_ge_3_count,rating_ge_3_rate,rating_ge_4_count,rating_ge_4_rate
0,163731,0.1637,424928,0.4248
1,836478,0.8363,575281,0.5752


## users_df preprocess

In [37]:
age_map = {
    1: "under_18",
    18: "18_24",
    25: "25_34",
    35: "35_44",
    45: "45_49",
    50: "50_55",
    56: "56_plus",
}

occupation_map = {
    0: "other_or_not_specified",
    1: "academic_educator",
    2: "artist",
    3: "clerical_admin",
    4: "college_grad_student",
    5: "customer_service",
    6: "doctor_health_care",
    7: "executive_managerial",
    8: "farmer",
    9: "homemaker",
    10: "k12_student",
    11: "lawyer",
    12: "programmer",
    13: "retired",
    14: "sales_marketing",
    15: "scientist",
    16: "self_employed",
    17: "technician_engineer",
    18: "tradesman_craftsman",
    19: "unemployed",
    20: "writer",
}

users_df_transformed = users_df.copy()
users_df_transformed["age_category"] = users_df_transformed["age"].map(age_map)
users_df_transformed["occupation_category"] = users_df_transformed["occupation"].map(occupation_map)

users_df_transformed["gender_F"] = (users_df_transformed["gender"] == "F").astype(int)
users_df_transformed["gender_M"] = (users_df_transformed["gender"] == "M").astype(int)

age_dummies = pd.get_dummies(users_df_transformed["age_category"], prefix="age").astype(int)
occupation_dummies = pd.get_dummies(users_df_transformed["occupation_category"], prefix="occupation").astype(int)

users_df_transformed = pd.concat(
    [users_df_transformed, age_dummies, occupation_dummies],
    axis=1,
)

users_df_transformed.head()

,user_id,gender,age,occupation,zip_code,age_category,occupation_category,gender_F,gender_M,age_18_24,...,occupation_other_or_not_specified,occupation_programmer,occupation_retired,occupation_sales_marketing,occupation_scientist,occupation_self_employed,occupation_technician_engineer,occupation_tradesman_craftsman,occupation_unemployed,occupation_writer
0,1,F,1,10,48067,under_18,k12_student,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,M,56,16,70072,56_plus,self_employed,0,1,0,...,0,0,0,0,0,1,0,0,0,0
2,3,M,25,15,55117,25_34,scientist,0,1,0,...,0,0,0,0,1,0,0,0,0,0
3,4,M,45,7,02460,45_49,executive_managerial,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,5,M,25,20,55455,25_34,writer,0,1,0,...,0,0,0,0,0,0,0,0,0,1


## transaction_dfの成形

transaction_df で使う列を、`user`、`movie`、`rating` の由来ごとに整理する。

In [38]:
user_attribute_columns = [
    "gender_F",
    "gender_M",
] + [
    column for column in users_df_transformed.columns
    if (
        column.startswith("age_") or column.startswith("occupation_")
    ) and column not in ["age_category", "occupation_category"]
]

movie_attribute_columns = [
    column for column in movies_df_transformed.columns
    if column.startswith("genre_") or column.startswith("release_decade_")
]

transaction_column_config = {
    "user_readable_columns": [
        "user_id",
        "gender",
        "age_category",
        "occupation_category",
    ],
    "user_attribute_columns": user_attribute_columns,
    "movie_readable_columns": [
        "movie_id",
        "title",
        "release_decade",
        "genres",
    ],
    "movie_attribute_columns": movie_attribute_columns,
    "rating_columns": [
        "user_id",
        "movie_id",
        "rating",
        "rating_ge_3",
        "rating_ge_4",
        "rating_datetime",
    ],
}

transaction_column_summary = pd.DataFrame(
    [
        {"source": source, "column_name": column}
        for source, columns in transaction_column_config.items()
        for column in columns
    ]
)

transaction_column_summary

,source,column_name
0,user_readable_columns,user_id
1,user_readable_columns,gender
2,user_readable_columns,age_category
3,user_readable_columns,occupation_category
4,user_attribute_columns,gender_F
...,...,...
67,rating_columns,movie_id
68,rating_columns,rating
69,rating_columns,rating_ge_3
70,rating_columns,rating_ge_4


In [39]:
transaction_df = (
    ratings_df_transformed[transaction_column_config["rating_columns"]]
    .merge(
        users_df_transformed[
            transaction_column_config["user_readable_columns"]
            + transaction_column_config["user_attribute_columns"]
        ],
        on="user_id",
        how="left",
    )
    .merge(
        movies_df_transformed[
            transaction_column_config["movie_readable_columns"]
            + transaction_column_config["movie_attribute_columns"]
        ],
        on="movie_id",
        how="left",
    )
)

transaction_df.head()

,user_id,movie_id,rating,rating_ge_3,rating_ge_4,rating_datetime,gender,age_category,occupation_category,gender_F,...,release_decade_1910s,release_decade_1920s,release_decade_1930s,release_decade_1940s,release_decade_1950s,release_decade_1960s,release_decade_1970s,release_decade_1980s,release_decade_1990s,release_decade_2000s
0,1,1193,5,1,1,2000-12-31 22:12:40,F,under_18,k12_student,1,...,0,0,0,0,0,0,1,0,0,0
1,1,661,3,1,0,2000-12-31 22:35:09,F,under_18,k12_student,1,...,0,0,0,0,0,0,0,0,1,0
2,1,914,3,1,0,2000-12-31 22:32:48,F,under_18,k12_student,1,...,0,0,0,0,0,1,0,0,0,0
3,1,3408,4,1,1,2000-12-31 22:04:35,F,under_18,k12_student,1,...,0,0,0,0,0,0,0,0,0,1
4,1,2355,5,1,1,2001-01-06 23:38:11,F,under_18,k12_student,1,...,0,0,0,0,0,0,0,0,1,0


## データ保存

各 transformed データフレームと transaction_df を `data` 配下へ parquet 形式で保存する。

In [40]:
movies_df_transformed.to_parquet(DATA_DIR / "movies_transformed.parquet", index=False)
ratings_df_transformed.to_parquet(DATA_DIR / "ratings_transformed.parquet", index=False)
users_df_transformed.to_parquet(DATA_DIR / "users_transformed.parquet", index=False)
transaction_df.to_parquet(DATA_DIR / "transaction.parquet", index=False)

pd.DataFrame(
    {
        "saved_file": [
            "movies_transformed.parquet",
            "ratings_transformed.parquet",
            "users_transformed.parquet",
            "transaction.parquet",
        ]
    }
)

,saved_file
0,movies_transformed.parquet
1,ratings_transformed.parquet
2,users_transformed.parquet
3,transaction.parquet
